# Chapter 16.3. ML2 총정리 — 하나의 MDP, 네 경로

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter16_3_one_mdp_four_paths.ipynb)

책 본문: [Section 16.3](https://smhanlab.com/book-ml/kor/ml2/chapter16/3.html)

이 노트북은 16.3절의 "같은 작은 MDP를 네 경로로"를 코드로 실행한다:
**① 벨만방정식 손계산 (Ch4) → ② 몬테카를로 (Ch5) → ③ SARSA (Ch6 on-policy) → ④ Q-learning (Ch6 off-policy)**.
목표는 책의 숫자를 완벽히 재현하는 것이 아니라 *구조*를 확인하는 것이다 —
네 경로가 "같은 답, 또는 설명 가능한 다른 답"에 도달하며,
갱신식의 `max` 한 줄 차이가 **수렴 대상 자체**를 바꾼다는 것.

## 0. MDP: 달리거나 쉬거나 (3-state, \(\gamma=0.9\))

| 상태 | 행동 | 전이 | 보상 |
|---|---|---|---|
| S (시작) | run | G (0.7) / S (0.3, 미끄러짐) | 0 |
| S (시작) | rest | S (남음) | −1 |
| G (골) | collect | T | +3 |
| T (터미널) | — | self-loop | 0 |

터미널은 보상 0의 self-loop로 표현한다 — Chapter 4 노트북과 같은 장치.

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

IMG = "/home/smhan/book-ml/kor/src/images"

# 상태: S=0(시작), G=1(골), T=2(터미널)
# 행동: 0=run/collect, 1=rest (S에서만)
gamma = 0.9

def step(s, a, rng):
    """(다음 상태, 보상) 반환."""
    if s == 0 and a == 0:                       # S: run
        return (1 if rng.random() < 0.7 else 0), 0.0
    if s == 0 and a == 1:                       # S: rest
        return 0, -1.0
    if s == 1:                                  # G: collect
        return 2, 3.0
    return 2, 0.0                               # T: self-loop

## 1. 경로 ① (모델 기반, Ch4): 벨만방정식 손계산

+3을 받는 유일한 길은 G에서 collect이므로 \(V^*(G)=3\). S에서는 run이 최선이라는
*후보*를 두고 벨만방정식을 푼다:

\[V^*(S) = 0.9\,(0.7 \times 3 + 0.3\,V^*(S)) = 1.89 + 0.27\,V^*(S)
\;\;\Longrightarrow\;\; V^*(S) = \tfrac{1.89}{0.73}\]

그리고 검증(후보+검증): \(Q^*(S,\text{run}) > Q^*(S,\text{rest})\)인가?

In [2]:
# 후보: greedy = (run@S, collect@G)
V_star_S = 1.89 / (1 - 0.27)
V_star_G = 3.0
Q_star_run = V_star_S
Q_star_rest = -1 + gamma * V_star_S
print(f"V*(S) = {V_star_S:.4f},  V*(G) = {V_star_G:.4f}")
print(f"Q*(S,run) = {Q_star_run:.4f}  >  Q*(S,rest) = {Q_star_rest:.4f}  (후보 성립)")

# value iteration으로 검증 (Ch4 코드와 같은 구조)
V = [0.0, 0.0, 0.0]
for _ in range(1000):
    V = [
        max(0.0 + gamma * (0.7 * V[1] + 0.3 * V[0]),    # S: run
            -1.0 + gamma * V[0]),                        # S: rest
        3.0 + gamma * V[2],                              # G: collect
        0.0,                                             # T: 터미널
    ]
print(f"value iteration: V(S) = {V[0]:.4f}, V(G) = {V[1]:.4f}")
assert abs(V[0] - V_star_S) < 1e-6 and abs(V[1] - V_star_G) < 1e-6
print("손계산과 value iteration이 일치.")

V*(S) = 2.5890,  V*(G) = 3.0000
Q*(S,run) = 2.5890  >  Q*(S,rest) = 1.3301  (후보 성립)
value iteration: V(S) = 2.5890, V(G) = 3.0000
손계산과 value iteration이 일치.


## 2. 경로 ② (모델 프리, Ch5): 몬테카를로

greedy 정책(run@S, collect@G)으로 에피소드를 끝까지 roll-out하고, *실제 실현된
리턴 자체*를 목표값으로 쓴다(첫방문 MC). MC는 **편향이 없다** — 실현 리턴의
평균의 기댓값이 정확히 \(V^\pi\)이기 때문에, "충분히 많이 돌리면"이라는
약속은 이 예에서 정말로 지켜진다.

In [3]:
rng = np.random.default_rng(42)

def mc_episode(rng):
    """greedy rollout 1회; S에서의 (첫방문) 리턴을 반환."""
    s, ret, disc = 0, 0.0, 1.0
    while s != 2:
        if s == 0:                      # run
            r = 0.0
            s = 1 if rng.random() < 0.7 else 0
        else:                           # s == 1: collect
            r, s = 3.0, 2
        ret += disc * r
        disc *= gamma
    return ret

n_ep = 4000
returns = np.array([mc_episode(rng) for _ in range(n_ep)])
V_mc = returns.mean()
print(f"MC ({n_ep} 에피소드): V(S) = {V_mc:.4f}   (참값 {V_star_S:.4f}, 차이 {abs(V_mc - V_star_S):.4f})")
assert abs(V_mc - V_star_S) < 0.05
print("MC는 편향이 없다 — 실현 리턴의 평균이 참값과 일치.")

MC (4000 에피소드): V(S) = 2.5942   (참값 2.5890, 차이 0.0052)
MC는 편향이 없다 — 실현 리턴의 평균이 참값과 일치.


## 3. 경로 ③ (Ch6, on-policy): SARSA

SARSA의 갱신은 **다음에 실제로 고른 행동**(ε-greedy 행동정책의 선택)을 쓴다 —
배우는 것은 \(V^*\)가 아니라 "행동정책의 값" \(V^\pi\)다.
\(\varepsilon=0.3\)를 *고정*하면, ε-greedy는 run을 \(0.7+0.15=0.85\), rest를 0.15로 고른다.
먼저 이 행동정책의 값을 해석적으로 구한다.

In [4]:
# 해석: ε=0.3 고정 ε-greedy의 값
eps = 0.3
p_run = (1 - eps) + eps / 2              # 0.85
# V = p_run*(1.89 + 0.27*V) + (1-p_run)*(-1 + 0.9*V)  를 V에 대해 풀면
V_pi_S = (2.89 * p_run - 1) / (0.1 + 0.63 * p_run)
Q_pi_run = 1.89 + 0.27 * V_pi_S
Q_pi_rest = -1 + 0.9 * V_pi_S
print(f"P(run) = {p_run:.2f},  P(rest) = {1 - p_run:.2f}")
print(f"V^π(S) = {V_pi_S:.4f}    (V*과의 간극 = {V_star_S - V_pi_S:.4f} = '탐험의 가격')")
print(f"Q^π(S,run) = {Q_pi_run:.4f},  Q^π(S,rest) = {Q_pi_rest:.4f}")

P(run) = 0.85,  P(rest) = 0.15
V^π(S) = 2.2919    (V*과의 간극 = 0.2971 = '탐험의 가격')
Q^π(S,run) = 2.5088,  Q^π(S,rest) = 1.0627


In [5]:
def run_sarsa(n_ep, eps0, alpha, eps_decay, seed):
    """SARSA; (최종 Q, 에피소드별 V^π(S) 궤적) 반환."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((3, 2))

    def act(s, eps):
        if rng.random() < eps:                                  # 탐험
            return int(rng.integers(2)) if s == 0 else 0
        return int(np.argmax(Q[s])) if s < 2 else 0             # greedy

    s, traj = 0, []
    a = act(s, eps0)
    for ep in range(n_ep):
        eps = eps0 * (eps_decay ** ep)
        while s != 2:
            ns, r = step(s, a, rng)
            na = act(ns, eps)                                   # on-policy: 실제로 고른 행동
            Q[s, a] += alpha * (r + gamma * Q[ns, na] - Q[s, a])
            s, a = ns, na
        p_r = (1 - eps) + eps / 2
        traj.append(p_r * Q[0, 0] + (1 - p_r) * Q[0, 1])        # ε-greedy 행동정책의 상태값
        s, a = 0, act(0, eps)
    return Q, np.array(traj)

Q_fix, traj_fix = run_sarsa(3000, 0.3, 0.1, 1.00, 42)
print(f"SARSA (ε=0.3 고정, 3000 에피소드):  V^π(S) = {traj_fix[-1]:.4f}   (해석값 {V_pi_S:.4f})")
assert abs(traj_fix[-1] - V_pi_S) < 0.15

Q_dec, traj_dec = run_sarsa(3000, 0.3, 0.1, 0.995, 42)
print(f"SARSA (ε 매 에피소드 0.995배 감쇠, 3000 에피소드): V^π(S) = {traj_dec[-1]:.4f}   (참값 {V_star_S:.4f})")
assert abs(traj_dec[-1] - V_star_S) < 0.15
print("ε를 감쇠하면 행동정책이 greedy에 수렴하므로, SARSA도 V*에 도달한다.")

SARSA (ε=0.3 고정, 3000 에피소드):  V^π(S) = 2.2815   (해석값 2.2919)
SARSA (ε 매 에피소드 0.995배 감쇠, 3000 에피소드): V^π(S) = 2.5530   (참값 2.5890)
ε를 감쇠하면 행동정책이 greedy에 수렴하므로, SARSA도 V*에 도달한다.


## 4. 경로 ④ (Ch6, off-policy): Q-learning

SARSA와 갱신식이 **한 줄**만 다르다 — "다음에 실제로 고른 행동"을
\(\max\)로 바꾼 것이다. "다음에서는 최선을 고른다면"을 목표값으로 삼으니
*행동정책과 무관하게* 최적 \(Q^*\)에 직접 수렴한다.

In [6]:
def run_qlearning(n_ep, eps0, eps_floor, alpha, seed):
    """Q-learning; (최종 Q, 에피소드별 Q(S,run) 궤적) 반환."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((3, 2))

    def act(s, eps):
        if s == 1:
            return 0
        if rng.random() < eps:                                  # 탐험
            return int(rng.integers(2))
        return int(np.argmax(Q[s]))                             # greedy

    s, traj = 0, []
    a = act(s, eps0)
    for ep in range(n_ep):
        eps = max(eps_floor, eps0 * 0.997 ** ep)
        while s != 2:
            ns, r = step(s, a, rng)
            Q[s, a] += alpha * (r + gamma * np.max(Q[ns]) - Q[s, a])   # <-- max 한 줄
            a = act(ns, eps)
            s = ns
        traj.append(Q[0, 0])
        s, a = 0, act(0, eps)
    return Q, np.array(traj)

Q_ql, traj_ql = run_qlearning(2000, 0.3, 0.05, 0.1, 42)
print(f"Q-learning (2000 에피소드, ε를 0.05까지 감쇠):")
print(f"  Q(S,run)     = {Q_ql[0, 0]:.4f}   (참값 {V_star_S:.4f})")
print(f"  Q(S,rest)    = {Q_ql[0, 1]:.4f}   (참값 {Q_star_rest:.4f})")
print(f"  Q(G,collect) = {Q_ql[1, 0]:.4f}   (참값 3.0000)")

Q-learning (2000 에피소드, ε를 0.05까지 감쇠):
  Q(S,run)     = 2.6059   (참값 2.5890)
  Q(S,rest)    = 1.3332   (참값 1.3301)
  Q(G,collect) = 3.0000   (참값 3.0000)


## 5. 수렴 비교

| 경로 | 무엇을 쓰는지 | \(V(S)\) 추정/수렴값 | 수렴 대상 |
|---|---|---|---|
| ① 벨만방정식 손계산 | 전이 모델 | 2.589 | 최적값 \(V^*\) (정확) |
| ② 몬테카를로 | 에피소드 전체 리턴 | ≈2.59 | greedy 정책의 값 (편향 없음) |
| ③ SARSA (ε=0.3 고정) | 1스텝 TD (on-policy) | ≈2.29 | *행동정책* \(V^\pi\) |
| ④ Q-learning | 1스텝 TD (off-policy) | ≈2.59 | 최적값 \(Q^*\) |

아래 학습 곡선은, "1스텝 TD 오차"라는 *같은 구조*를 써도 ③은 행동정책의 값에,
②/④는 최적값에 수렴함을 보여준다.

In [7]:
def ema(x, a=0.01):
    """지수이동평균 — 곡선을 매끄럽게 하되 수렴 *높이*를 가리지 않는다."""
    out = np.empty_like(x, dtype=float)
    out[0] = x[0]
    for i in range(1, len(x)):
        out[i] = a * x[i] + (1 - a) * out[i - 1]
    return out

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axhline(V_star_S, color="tab:red", ls="--", lw=1.2,
           label=f"① V*(S) = {V_star_S:.3f} (Bellman eq.)")
ax.axhline(V_pi_S, color="tab:orange", ls="--", lw=1.2,
           label=f"V^π(S) = {V_pi_S:.3f} (ε=0.3 behavior policy)")
ax.plot(ema(returns), color="tab:blue", lw=1.3, label="② MC (greedy)")
ax.plot(ema(traj_fix), color="tab:green", lw=1.3, label="③ SARSA (ε=0.3 fixed)")
ax.plot(ema(traj_ql), color="tab:purple", lw=1.3, label="④ Q-learning (Q(S,run))")
ax.set_title("One MDP, four paths — the convergence target depends on 'what you ask'")
ax.set_xlabel("Episode")
ax.set_ylabel("State value of S")
ax.set_xlim(0, 4000)
ax.set_ylim(0, 3.4)
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
fig.savefig(IMG + "/ch16_3_four_paths_curves.svg", bbox_inches="tight")
plt.show()

**읽기**: ③과 ④의 간극 \(V^* - V^\pi \approx 0.297\)은 버그가 아니라 *기능*이다 —
"ε-greedy가 30%의 시간을 미끄러지며 낭비한다"는 사실이 가치에 반영된 숫자다.
`max` 한 줄은 갱신을 바꾸는 것을 넘어 **수렴 대상 자체**를 바꾼다.
ε를 감쇠하면(매 에피소드 0.995배) 행동정책이 greedy에 가까워지고 ③도
2.589로 끝난다 — 탐험 예산과 수렴 대상은 연결되어 있다.